# Langchain VectorDB Retriever

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['HF_TOKEN']=os.getenv('HF_TOKEN')

---

## Load - Generate Sample Document (Splitted)

In [3]:
from langchain_core.documents import Document

In [4]:
document=[
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source":"fish-pets-doc"}
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"}
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source":"mammal-pets-doc"}
    ),
]

In [5]:
document

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

### Set up LLM

In [9]:
from langchain_groq import ChatGroq

In [10]:
llm=ChatGroq(model='llama-3.1-8b-instant')

In [11]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7289dd1adb90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7289dd0a5890>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

## Embed - HuggingFaceEmbeddings()

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

In [14]:
hugginface_embeddings = HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

/home/iceyisaak/anaconda3/envs/langchain-vectordb-retriever/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
hugginface_embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## Store - ChromaDB

In [6]:
from langchain_chroma import Chroma

In [16]:
vector_db = Chroma.from_documents(document,hugginface_embeddings)

In [17]:
vector_db

---

## Query - .asimilarity_search()

In [24]:
similar_search = vector_db.similarity_search('cat')

In [25]:
similar_search

[Document(id='ee557afb-4c38-4276-a167-6ea854c0414f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='b33952c3-1144-4de6-a131-b919b81c63f0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='26fddb2c-3539-4ed3-98f8-fd6871d6d616', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='f1cb5c85-d80e-4088-bfde-6522f22e918d', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [26]:
asimilar_search = vector_db.asimilarity_search('cat')

In [27]:
asimilar_search

<coroutine object VectorStore.asimilarity_search at 0x728a51d2d300>

### Async Query

In [28]:
await asimilar_search

[Document(id='ee557afb-4c38-4276-a167-6ea854c0414f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='b33952c3-1144-4de6-a131-b919b81c63f0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='26fddb2c-3539-4ed3-98f8-fd6871d6d616', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='f1cb5c85-d80e-4088-bfde-6522f22e918d', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

### Similarity Search with Score

In [29]:
sim_search_with_score = vector_db.similarity_search_with_score('cat')

In [30]:
sim_search_with_score

[(Document(id='ee557afb-4c38-4276-a167-6ea854c0414f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.935105562210083),
 (Document(id='b33952c3-1144-4de6-a131-b919b81c63f0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740900039672852),
 (Document(id='26fddb2c-3539-4ed3-98f8-fd6871d6d616', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.595690369606018),
 (Document(id='f1cb5c85-d80e-4088-bfde-6522f22e918d', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.6657925844192505)]

---

## Retrievers

In [31]:
from typing import List

In [32]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

In [33]:
retriever=RunnableLambda(vector_db.similarity_search).bind(k=1)

In [34]:
retriever.batch(["cat","dog"])

[[Document(id='ee557afb-4c38-4276-a167-6ea854c0414f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='b33952c3-1144-4de6-a131-b919b81c63f0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

---

### Vector .as_retriever() - Best way to query from VectorDB

In [37]:
vectorstore_retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

In [38]:
vectorstore_retriever.batch(["cat","dog"])

[[Document(id='ee557afb-4c38-4276-a167-6ea854c0414f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='b33952c3-1144-4de6-a131-b919b81c63f0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

---

## RAG - VectorStore with Prompt Template

In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [40]:
message = """
    Answer this question using the provided context only.

    {question}

    Context:

    {context}
"""

In [41]:
prompt = ChatPromptTemplate.from_messages([("human",message)])

In [42]:
rag_chain={
    "context":vectorstore_retriever,
    "question":RunnablePassthrough()
}|prompt|llm

In [ ]:
response=rag_chain.invoke("Tell me about cats.")

In [46]:
response.content

'Cats are independent pets that often enjoy their own space.'

---